# DRB2 (A) x DRB4 (B) Domain Contact Analysis -- per pose cluster

**Kernel:** `abcfold-drbs-notebook` (`envs/notebook.yaml`) -- install once:
```
conda env create -f envs/notebook.yaml
conda activate abcfold-drbs-notebook
python -m ipykernel install --user --name abcfold-drbs-notebook
```

PLIP contacts from `results/drb2_drb4/all_selected_summary.csv`, produced by
`worflows/postprocessing/Snakefile` (stage 3g `run_plip` + 3h `aggregate`) --
the standalone DRB2-DRB4 binary complex (no DCL4, no dsRNA), receptor = DRB2
(chain A), ligand = DRB4 (chain B) (`configs/drb2_drb4.yaml`'s `plip.chains`).

**Data provenance quirk worth knowing:** `scripts/aggregate_summaries.py` was
inherited verbatim from the sibling `ab_initio_modelling_drbs_dcl4_ds_rna_complexes`
project, where its `extract_metadata()` reads `parts[-5]` as "replica" and
`parts[-4]` as "model". This pipeline's PLIP output path is one level deeper
than that project's (`results/<complex>/plip/<cluster>/<fname>/<fname>_report/csv/summary.csv`
vs `results/<protein>/<replica>/<model>/<model>_report/csv/summary.csv`), and
the two structures happen to line up: the "replica" column this script writes
is actually this pipeline's pose **cluster** number, and "model" is the staged
**fname** (`rank_<NN>_<backend>_seed<N>_sample<M>`). We rename both below for
clarity -- no data is wrong, just relabeled at aggregation time.

This notebook does **not** re-derive pose clusters -- those come from the
rigid-anchor Kabsch + hierarchical-RMSD clustering already computed by
`scripts/pose_cluster_anchor.py` (`results/drb2_drb4/pose_clusters.csv`,
explored interactively in `notebooks/pose_clustering.ipynb`). Here we stratify
**PLIP domain contacts** by that same cluster label.

Domain boundaries (1-based inclusive -- identical residue numbering to the
sibling project since sequences were copied verbatim, see
`configs/drb2_drb4.yaml`). The original boundaries (ChimeraX coloring
script, inherited from the sibling project) were computed from **UniProt's
PROSITE domain annotation** -- a sequence-profile-based domain call, not
itself informed by any of the structural/disorder-predictor evidence this
notebook generates.

**Correction applied here:** PROSITE's DRB2 dsRBD2/disordered boundary
(87-155 / 156-434) undercalls the folded region. The "Boltz-2 over-folding
investigation" section below originally found residues 156-188 forming
near-universal (~100%%, all 5 surviving backends including the
well-behaved alphafold3/protenix) helix with no exception -- structural
evidence alone was suggestive but not conclusive (a real coupled
folding-binding element would look similar). Cross-checked against
sequence-only disorder predictors AIUpred/ANCHOR2 (see the DRB2/DRB4
fold-upon-binding review, `tools/MC2` MoRFchibi 2.0 + AIUpred web results in
`data/fold_inputs/drb2_drb4/`): AIUpred's own disorder score for 156-188 is
low (~0.09-0.47, i.e. predicted *ordered*, not disordered) -- inconsistent
with a genuine disorder-to-order MoRF (those score *high* disorder alone,
per DRB2's real MoRF candidate at 425-434, disorder ~0.88). The AIUpred
disorder score and the cross-backend helix frequency both transition
together, sharply, at **residue 189** (disorder crosses back above ~0.5 and
helix frequency collapses from ~98%% at 188 to ~0%% by 199) -- residues
156-188 are correctly folded, not a fold-upon-binding event, just
under-called by the original PROSITE-based boundary. dsRBD2 is extended to
188 and disordered now starts at 189 below; DRB4's boundaries were checked
too and are not revised -- the equivalent evidence there (see the same
review) supports the original disordered/cryoEM_domain split rather than
contradicting it.

**DRB2 (chain A)**

| DRB2 domain | Residues |
|---|---|
| dsRBD1 | 1-70 |
| linker | 71-86 |
| dsRBD2 | 87-188 (was 87-155 under the original PROSITE-based call) |
| disordered | 189-434 (was 156-434) |

**DRB4 (chain B)**

| DRB4 domain | Residues |
|---|---|
| dsRBD1 | 4-73 |
| linker | 74-81 |
| dsRBD2 | 82-150 |
| disordered | 151-291 |
| cryoEM_domain | 292-355 |


In [1]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from pathlib import Path

ROOT = Path("..")
RESULTS_DIR = ROOT / "results" / "drb2_drb4"
RECEPTOR_CHAIN, RECEPTOR_NAME = "A", "DRB2"
LIGAND_CHAIN, LIGAND_NAME = "B", "DRB4"
FIGURES_DIR = RESULTS_DIR / "figures" / "domain_analysis"

TEMPLATE = "plotly_white"
CLUSTER_PALETTE = px.colors.qualitative.Set1

def out_path(subdir, filename):
    out_dir = FIGURES_DIR / subdir
    out_dir.mkdir(parents=True, exist_ok=True)
    return out_dir / filename

def save_fig(fig, filename, subdir=""):
    out = out_path(subdir, filename)
    fig.write_html(out, include_plotlyjs="cdn")
    print(f"Saved: {out}")
    fig.show()


## Load data

In [2]:
csv_path = RESULTS_DIR / "all_selected_summary.csv"
df = pd.read_csv(csv_path)
df = df.rename(columns={"replica": "cluster", "model": "fname"})
df["cluster"] = df["cluster"].astype(int)

# Bring in per-model metadata (backend, seed, ranking_score, ptm, iptm) staged
# by scripts/select_top_n_per_cluster.py -- join key is the staged cif's stem,
# which is exactly the "fname" aggregate_summaries.py recovered above.
sel = pd.read_csv(RESULTS_DIR / "selected_models.csv")
sel["fname"] = sel["staged_cif"].apply(lambda p: Path(p).stem)
sel = sel[["fname", "cluster", "backend", "seed", "sample_index", "ranking_score", "ptm", "iptm"]]

df = df.merge(sel, on=["fname", "cluster"], how="left", validate="many_to_one")
n_missing_meta = df["backend"].isna().sum()
if n_missing_meta:
    print(f"WARNING: {n_missing_meta} contact rows have no matching selected_models.csv "
          f"entry (stale/partial aggregate CSV -- re-run after the full postprocessing "
          f"run finishes)")

n_models = df.groupby(["cluster", "fname"]).ngroups
print(f"{csv_path}: {len(df)} contact rows, {df['cluster'].nunique()} pose cluster(s), "
      f"{n_models} model(s)")
df.groupby("cluster")["fname"].nunique().rename("n_models").to_frame()


../results/drb2_drb4/all_selected_summary.csv: 246768 contact rows, 2 pose cluster(s), 598 model(s)


,n_models
cluster,
1,597
2,1


## Filter out numerically-unconverged structures

Some minimized structures have wildly nonphysical final energies -- e.g. one
chai1 structure at +4x10^8 kJ/mol -- genuine minimization failures that
`minimize_cif.py`'s `nan`-only divergence check doesn't catch, since these
are real (if absurd) numbers, not literally `nan`. Applied here, right after
loading, so **every** section below (domain heatmaps, per-cluster,
per-backend, the Boltz-2 investigation) already works from the cleaned set --
this is also why rosettafold3 no longer needs any special-case handling
further down: if its structures survive this filter they're treated exactly
like any other backend's, and if they don't, they're simply absent, the same
as any other backend's failed structures would be.


In [3]:
def read_final_energy(energy_csv_path):
    """Last row's energy_kJ_mol from a minimize_cif.py *_energy.csv sidecar."""
    if not energy_csv_path.exists():
        return np.nan
    last_energy = np.nan
    with open(energy_csv_path) as fh:
        next(fh, None)  # header
        for line in fh:
            _, e = line.strip().split(",")
            last_energy = float(e)
    return last_energy

energy_rows = []
for pdb_path in sorted(RESULTS_DIR.glob("minimized/*/*/*.pdb")):
    if pdb_path.stem.endswith(("_fixed", "_amber")):
        continue
    cluster = int(pdb_path.parent.parent.name)
    energy_csv = pdb_path.with_name(pdb_path.stem + "_energy.csv")
    energy_rows.append({"fname": pdb_path.stem, "cluster": cluster,
                         "final_energy": read_final_energy(energy_csv)})

energy_df = pd.DataFrame(energy_rows).dropna(subset=["final_energy"])

# Robust (median / median-absolute-deviation) modified z-score (Iglewicz &
# Hoaglin) -- robust to the outliers themselves: a plain mean/std would be
# destroyed by a +4x10^8 value sitting next to a cluster of ~-70,000 ones,
# a MAD-based score isn't.
MOD_Z_THRESHOLD = 3.5

pooled_median = energy_df["final_energy"].median()
pooled_mad = (energy_df["final_energy"] - pooled_median).abs().median()
energy_df["energy_mod_z"] = 0.6745 * (energy_df["final_energy"] - pooled_median) / pooled_mad
energy_df["energy_ok"] = energy_df["energy_mod_z"].abs() <= MOD_Z_THRESHOLD

flagged = energy_df.loc[~energy_df["energy_ok"]].merge(
    sel[["fname", "cluster", "backend"]], on=["fname", "cluster"], how="left"
)
print(f"Pooled final-energy median={pooled_median:,.0f} kJ/mol, MAD={pooled_mad:,.0f}")
print(f"Flagged {len(flagged)} / {len(energy_df)} minimized model(s) as numerically "
      f"unconverged (|modified z-score| > {MOD_Z_THRESHOLD}), by backend:")
display(flagged.groupby("backend").size().rename("n_excluded").to_frame())


Pooled final-energy median=-72,523 kJ/mol, MAD=9,756
Flagged 57 / 546 minimized model(s) as numerically unconverged (|modified z-score| > 3.5), by backend:


,n_excluded
backend,
boltz,1
chai1,1
protenix,2
rosettafold3,53


In [4]:
# Apply the filter to the contacts dataframe -- everything below this cell
# works from the cleaned `df`.
good_pairs = set(zip(
    energy_df.loc[energy_df["energy_ok"], "cluster"],
    energy_df.loc[energy_df["energy_ok"], "fname"],
))
df_keys = pd.MultiIndex.from_arrays([df["cluster"], df["fname"]])
n_models_before = df.groupby(["cluster", "fname"]).ngroups
df = df[df_keys.isin(good_pairs)].copy()
n_models_after = df.groupby(["cluster", "fname"]).ngroups

print(f"{n_models_after} / {n_models_before} model(s) kept after the energy filter")
df.groupby("cluster")["fname"].nunique().rename("n_models").to_frame()


473 / 598 model(s) kept after the energy filter


,n_models
cluster,
1,472
2,1


## Domain definitions

In [5]:
# DRB2's dsRBD2/disordered boundary corrected from the original PROSITE-based
# call (87-155 / 156-434) to 87-188 / 189-434 -- see the intro cell for the
# evidence (near-universal cross-backend helix + AIUpred disorder score both
# transitioning sharply at 189, not 156). DRB4's boundaries are unrevised.
DRB2_DOMAINS = [
    ("dsRBD1", 1, 70),
    ("linker", 71, 86),
    ("dsRBD2", 87, 188),
    ("disordered", 189, 434),
]

DRB4_DOMAINS = [
    ("dsRBD1", 4, 73),
    ("linker", 74, 81),
    ("dsRBD2", 82, 150),
    ("disordered", 151, 291),
    ("cryoEM_domain", 292, 355),
]

def make_domain_mapper(domain_ranges):
    """domain_ranges: list of (label, start, end), inclusive on both ends.
    Returns a function mapping a pandas Series of residue numbers to domain
    labels; residues outside every range become NaN (reported separately)."""
    intervals = pd.IntervalIndex.from_tuples(
        [(start, end) for _, start, end in domain_ranges], closed="both"
    )
    labels = [label for label, _, _ in domain_ranges]

    def mapper(resnr_series):
        idx = intervals.get_indexer(resnr_series.astype(float))
        return pd.Series(
            [labels[i] if i != -1 else pd.NA for i in idx],
            index=resnr_series.index, dtype="object",
        )
    return mapper

DRB2_LABELS = [l for l, _, _ in DRB2_DOMAINS]
DRB4_LABELS = [l for l, _, _ in DRB4_DOMAINS]
drb2_mapper = make_domain_mapper(DRB2_DOMAINS)
drb4_mapper = make_domain_mapper(DRB4_DOMAINS)

df["drb2_domain"] = drb2_mapper(df["resnr"])
df["drb4_domain"] = drb4_mapper(df["resnr_lig"])

n_unmapped_r = df["drb2_domain"].isna().sum()
n_unmapped_l = df["drb4_domain"].isna().sum()
print(f"Unmapped DRB2 residues (outside any domain): {n_unmapped_r} ({100*n_unmapped_r/len(df):.1f}%)")
print(f"Unmapped DRB4 residues (outside any domain): {n_unmapped_l} ({100*n_unmapped_l/len(df):.1f}%)")
df[["resnr", "drb2_domain", "resnr_lig", "drb4_domain"]].head(5)


Unmapped DRB2 residues (outside any domain): 0 (0.0%)
Unmapped DRB4 residues (outside any domain): 275 (0.7%)


,resnr,drb2_domain,resnr_lig,drb4_domain
221,25,dsRBD1,22,dsRBD1
222,53,dsRBD1,36,dsRBD1
223,53,dsRBD1,56,dsRBD1
224,77,linker,342,cryoEM_domain
225,78,linker,342,cryoEM_domain


## Contacts per domain (all clusters pooled)

In [6]:
def domain_proportions(data, domain_col, ordered_labels, protein_label, title_suffix="", subdir="domain_contacts"):
    counts = data[domain_col].value_counts()
    props = pd.Series({d: counts.get(d, 0) for d in ordered_labels})
    props_pct = props / props.sum() * 100 if props.sum() else props

    fig = go.Figure(go.Bar(
        x=ordered_labels, y=props_pct.values,
        text=[f"{v:.1f}%" if v > 1 else "" for v in props_pct.values],
        textposition="outside",
        marker_color="#4C72B0",
        hovertemplate="%{x}<br>%{y:.2f}% of contacts<extra></extra>",
    ))
    fig.update_layout(
        title=f"Contact proportion per {protein_label} domain{title_suffix}",
        xaxis_title=f"{protein_label} domain", yaxis_title="Proportion of contacts (%)",
        template=TEMPLATE, width=max(500, len(ordered_labels) * 90), height=420,
    )
    save_fig(fig, f"{protein_label}_domain_proportions.html", subdir)
    return props

props_receptor = domain_proportions(df, "drb2_domain", DRB2_LABELS, "DRB2")
props_ligand   = domain_proportions(df, "drb4_domain", DRB4_LABELS, "DRB4")


Saved: ../results/drb2_drb4/figures/domain_analysis/domain_contacts/DRB2_domain_proportions.html


Saved: ../results/drb2_drb4/figures/domain_analysis/domain_contacts/DRB4_domain_proportions.html


## DRB2 x DRB4 domain-pair heatmap (all clusters pooled)

In [7]:
def domain_pair_heatmap(data, title, filename, n_models_norm, subdir="domain_contacts"):
    sub = data.dropna(subset=["drb2_domain", "drb4_domain"])
    ct = (
        sub.groupby(["drb2_domain", "drb4_domain"], observed=True)
        .size()
        .unstack(fill_value=0)
        .reindex(index=DRB2_LABELS, columns=DRB4_LABELS, fill_value=0)
    )
    rate = ct / n_models_norm if n_models_norm else ct

    fig = go.Figure(go.Heatmap(
        z=rate.values, x=DRB4_LABELS, y=DRB2_LABELS,
        colorscale="YlOrRd",
        text=[[f"{v:.2f}" if v > 0 else "" for v in row] for row in rate.values],
        texttemplate="%{text}", textfont=dict(size=10),
        hovertemplate="DRB2 domain: %{y}<br>DRB4 domain: %{x}<br>%{z:.3f} contacts/model<extra></extra>",
        colorbar=dict(title="Mean<br>contacts/<br>model"),
    ))
    fig.update_layout(
        title=title, xaxis_title="DRB4 domain", yaxis_title="DRB2 domain",
        yaxis=dict(autorange="reversed"), template=TEMPLATE,
        width=max(500, len(DRB4_LABELS) * 90), height=max(420, len(DRB2_LABELS) * 55),
    )
    save_fig(fig, filename, subdir)
    return ct

n_models_total = df.groupby(["cluster", "fname"]).ngroups
ct_all = domain_pair_heatmap(
    df, "DRB2 x DRB4 domain contact pairs (all clusters) -- mean contacts per model",
    "drb2_drb4_domain_heatmap_all_clusters.html", n_models_total,
)
ct_all


Saved: ../results/drb2_drb4/figures/domain_analysis/domain_contacts/drb2_drb4_domain_heatmap_all_clusters.html


drb4_domain,dsRBD1,linker,dsRBD2,disordered,cryoEM_domain
drb2_domain,,,,,
dsRBD1,2285,59,1251,3038,1453
linker,246,21,105,797,1231
dsRBD2,1576,42,1854,3570,3748
disordered,3350,335,3560,10385,2963


## Interaction-type breakdown (all clusters pooled)

In [8]:
print("Interaction type counts:")
display(df["interaction_type"].value_counts().rename("count").to_frame())

sub = df.dropna(subset=["drb2_domain", "drb4_domain"])
print("\nInteraction type by domain pair (top 20):")
itype_by_domain = (
    sub.groupby(["drb2_domain", "drb4_domain", "interaction_type"], observed=True)
    .size()
    .reset_index(name="count")
    .sort_values("count", ascending=False)
)
display(itype_by_domain.head(20))


Interaction type counts:


,count
interaction_type,
hydrogen_bonds,21669
hydrophobic_interactions,15176
salt_bridges,5061
pi-cation_interactions,206
pi-stacking,32



Interaction type by domain pair (top 20):


,drb2_domain,drb4_domain,interaction_type,count
5,disordered,disordered,hydrogen_bonds,5521
6,disordered,disordered,hydrophobic_interactions,3951
48,dsRBD2,disordered,hydrogen_bonds,1902
27,dsRBD1,disordered,hydrogen_bonds,1803
10,disordered,dsRBD1,hydrogen_bonds,1617
16,disordered,dsRBD2,hydrophobic_interactions,1606
43,dsRBD2,cryoEM_domain,hydrogen_bonds,1562
0,disordered,cryoEM_domain,hydrogen_bonds,1539
44,dsRBD2,cryoEM_domain,hydrophobic_interactions,1510
15,disordered,dsRBD2,hydrogen_bonds,1441


## Top contacted residue pairs (all clusters pooled)

In [9]:
def top_residue_pairs(data, n_models_norm):
    top = (
        data.assign(model_key=list(zip(data["cluster"], data["fname"])))
        .groupby(["resnr", "restype", "resnr_lig", "restype_lig"])["model_key"]
        .nunique()
        .rename("n_models")
        .reset_index()
        .rename(columns={"resnr": "drb2_resnr", "restype": "drb2_restype",
                          "resnr_lig": "drb4_resnr", "restype_lig": "drb4_restype"})
    )
    itypes = (
        data.groupby(["resnr", "restype", "resnr_lig", "restype_lig"])["interaction_type"]
        .agg(lambda s: ", ".join(sorted(s.unique())))
        .rename("interaction_types")
        .reset_index()
        .rename(columns={"resnr": "drb2_resnr", "restype": "drb2_restype",
                          "resnr_lig": "drb4_resnr", "restype_lig": "drb4_restype"})
    )
    top = top.merge(itypes, on=["drb2_resnr", "drb2_restype", "drb4_resnr", "drb4_restype"])
    top = top.sort_values("n_models", ascending=False)
    top["fraction_models"] = top["n_models"] / n_models_norm
    return top

top_pairs = top_residue_pairs(df, n_models_total)
out_csv = out_path("domain_contacts", "drb2_drb4_top_residue_contacts_all_clusters.csv")
top_pairs.to_csv(out_csv, index=False)
print(f"Top 20 residue pairs by number of models where the contact is observed "
      f"(out of {n_models_total} models). Saved: {out_csv}")
top_pairs.head(20)


Top 20 residue pairs by number of models where the contact is observed (out of 473 models). Saved: ../results/drb2_drb4/figures/domain_analysis/domain_contacts/drb2_drb4_top_residue_contacts_all_clusters.csv


,drb2_resnr,drb2_restype,drb4_resnr,drb4_restype,n_models,interaction_types,fraction_models
8520,180,ARG,344,ASP,95,salt_bridges,0.200846
8563,183,ILE,347,PHE,92,hydrophobic_interactions,0.194503
5403,98,ARG,325,CYS,86,hydrogen_bonds,0.181818
8523,180,ARG,347,PHE,79,"hydrogen_bonds, hydrophobic_interactions, pi-c...",0.167019
2025,28,ASP,6,LYS,79,"hydrogen_bonds, hydrophobic_interactions, salt...",0.167019
1832,24,ARG,60,GLU,78,salt_bridges,0.164905
8415,176,VAL,323,VAL,78,hydrophobic_interactions,0.164905
1714,22,CYS,25,ASN,74,hydrogen_bonds,0.156448
1823,24,ARG,23,TYR,74,"hydrogen_bonds, hydrophobic_interactions, pi-c...",0.156448
1561,20,TYR,27,ARG,73,"hydrogen_bonds, hydrophobic_interactions",0.154334


## Per-cluster domain breakdown

Everything above pools all pose clusters together. Here we repeat the same
domain-contact analysis **stratified by pose cluster** (`cluster` column,
straight from `results/drb2_drb4/pose_clusters.csv` via `select_top_n_per_cluster.py`
-- see the provenance note in the intro cell) to see whether different 3-D
docking poses correspond to different domain-domain interfaces.


In [10]:
clusters = sorted(df["cluster"].unique())
cluster_n_models = df.groupby("cluster").apply(lambda g: g["fname"].nunique(), include_groups=False)
print(f"{len(clusters)} pose cluster(s): " +
      ", ".join(f"cluster {c} (n={cluster_n_models[c]} models)" for c in clusters))


2 pose cluster(s): cluster 1 (n=472 models), cluster 2 (n=1 models)


In [11]:
# One DRB2 x DRB4 domain heatmap per cluster (own color scale per cluster, since
# cluster sizes -- and therefore mean contacts/model -- can differ a lot).
per_cluster_ct = {}
for c in clusters:
    sub_c = df[df["cluster"] == c]
    ct_c = domain_pair_heatmap(
        sub_c, f"DRB2 x DRB4 domain contact pairs -- cluster {c} (n={cluster_n_models[c]} models)",
        f"drb2_drb4_domain_heatmap_cluster{c}.html", cluster_n_models[c],
        subdir="per_cluster",
    )
    per_cluster_ct[c] = ct_c


Saved: ../results/drb2_drb4/figures/domain_analysis/per_cluster/drb2_drb4_domain_heatmap_cluster1.html


Saved: ../results/drb2_drb4/figures/domain_analysis/per_cluster/drb2_drb4_domain_heatmap_cluster2.html


In [12]:
# Side-by-side small-multiple comparison, one panel per cluster, shared color scale
# so panels are directly comparable at a glance.
zmax = max((ct / cluster_n_models[c]).values.max() for c, ct in per_cluster_ct.items())

fig = make_subplots(
    rows=1, cols=len(clusters), subplot_titles=[f"cluster {c} (n={cluster_n_models[c]})" for c in clusters],
    shared_yaxes=True,
)
for i, c in enumerate(clusters, start=1):
    rate = per_cluster_ct[c] / cluster_n_models[c]
    fig.add_trace(
        go.Heatmap(
            z=rate.values, x=DRB4_LABELS, y=DRB2_LABELS, colorscale="YlOrRd",
            zmin=0, zmax=zmax, showscale=(i == len(clusters)),
            text=[[f"{v:.2f}" if v > 0 else "" for v in row] for row in rate.values],
            texttemplate="%{text}", textfont=dict(size=9),
            hovertemplate=f"cluster {c}<br>DRB2: %{{y}}<br>DRB4: %{{x}}<br>%{{z:.3f}} contacts/model<extra></extra>",
            colorbar=dict(title="Mean<br>contacts/<br>model"),
        ),
        row=1, col=i,
    )
fig.update_yaxes(autorange="reversed")
fig.update_layout(
    title="DRB2 x DRB4 domain contact pairs by pose cluster",
    template=TEMPLATE, width=max(900, 320 * len(clusters)), height=460,
)
save_fig(fig, "drb2_drb4_domain_heatmap_by_cluster_sidebyside.html", "per_cluster")


Saved: ../results/drb2_drb4/figures/domain_analysis/per_cluster/drb2_drb4_domain_heatmap_by_cluster_sidebyside.html


In [13]:
# Domain-proportion comparison across clusters (grouped bars).
def domain_proportions_by_cluster(domain_col, ordered_labels, protein_label):
    rows = []
    for c in clusters:
        sub_c = df[df["cluster"] == c]
        counts = sub_c[domain_col].value_counts()
        total = counts.sum()
        for d in ordered_labels:
            rows.append({"cluster": f"cluster {c}", "domain": d,
                         "pct": 100 * counts.get(d, 0) / total if total else 0.0})
    plot_df = pd.DataFrame(rows)

    fig = px.bar(
        plot_df, x="domain", y="pct", color="cluster", barmode="group",
        category_orders={"domain": ordered_labels},
        color_discrete_sequence=CLUSTER_PALETTE,
        labels={"pct": "Proportion of contacts (%)", "domain": f"{protein_label} domain"},
        title=f"{protein_label} domain contact proportion by pose cluster",
    )
    fig.update_layout(template=TEMPLATE, width=max(600, len(ordered_labels) * 120), height=440)
    save_fig(fig, f"{protein_label}_domain_proportions_by_cluster.html", "per_cluster")
    return plot_df

_ = domain_proportions_by_cluster("drb2_domain", DRB2_LABELS, "DRB2")
_ = domain_proportions_by_cluster("drb4_domain", DRB4_LABELS, "DRB4")


Saved: ../results/drb2_drb4/figures/domain_analysis/per_cluster/DRB2_domain_proportions_by_cluster.html


Saved: ../results/drb2_drb4/figures/domain_analysis/per_cluster/DRB4_domain_proportions_by_cluster.html


In [14]:
print("Interaction type counts by cluster:")
display(
    df.groupby(["cluster", "interaction_type"], observed=True)
    .size()
    .rename("count")
    .reset_index()
    .pivot(index="interaction_type", columns="cluster", values="count")
    .fillna(0).astype(int)
)


Interaction type counts by cluster:


cluster,1,2
interaction_type,,
hydrogen_bonds,21624,45
hydrophobic_interactions,15143,33
pi-cation_interactions,206,0
pi-stacking,32,0
salt_bridges,5053,8


In [15]:
# Top 10 residue-pair contacts per cluster.
per_cluster_top = []
for c in clusters:
    sub_c = df[df["cluster"] == c]
    top_c = top_residue_pairs(sub_c, cluster_n_models[c])
    top_c.insert(0, "cluster", c)
    per_cluster_top.append(top_c.head(10))

per_cluster_top_df = pd.concat(per_cluster_top, ignore_index=True)
out_csv = out_path("per_cluster", "drb2_drb4_top_residue_contacts_by_cluster.csv")
per_cluster_top_df.to_csv(out_csv, index=False)
print(f"Saved: {out_csv}")
per_cluster_top_df


Saved: ../results/drb2_drb4/figures/domain_analysis/per_cluster/drb2_drb4_top_residue_contacts_by_cluster.csv


,cluster,drb2_resnr,drb2_restype,drb4_resnr,drb4_restype,n_models,interaction_types,fraction_models
0,1,180,ARG,344,ASP,95,salt_bridges,0.201271
1,1,183,ILE,347,PHE,92,hydrophobic_interactions,0.194915
2,1,98,ARG,325,CYS,86,hydrogen_bonds,0.182203
3,1,180,ARG,347,PHE,79,"hydrogen_bonds, hydrophobic_interactions, pi-c...",0.167373
4,1,28,ASP,6,LYS,79,"hydrogen_bonds, hydrophobic_interactions, salt...",0.167373
5,1,176,VAL,323,VAL,78,hydrophobic_interactions,0.165254
6,1,24,ARG,60,GLU,78,salt_bridges,0.165254
7,1,24,ARG,23,TYR,74,"hydrogen_bonds, hydrophobic_interactions, pi-c...",0.156780
8,1,22,CYS,25,ASN,74,hydrogen_bonds,0.156780
9,1,20,TYR,27,ARG,73,"hydrogen_bonds, hydrophobic_interactions",0.154661


## Per-backend domain breakdown

Same domain-contact analysis, but stratified by which ABCfold **backend**
produced each model (`backend` column, joined in from `selected_models.csv`)
instead of by pose cluster -- do the different architectures (AlphaFold3,
Boltz-2, Chai-1, OpenFold3, Protenix, RosettaFold3) agree on which domains
contact each other, or does one stand out? Note this is a genuinely different
grouping from the pose clusters above -- a backend can contribute models to
more than one cluster, and a cluster can be dominated by one backend, so
don't read the two sections as the same split relabelled.


In [16]:
backends = sorted(df["backend"].dropna().unique())
backend_n_models = df.groupby("backend").apply(lambda g: g["fname"].nunique(), include_groups=False)
print(f"{len(backends)} backend(s): " +
      ", ".join(f"{b} (n={backend_n_models[b]} models)" for b in backends))


5 backend(s): alphafold3 (n=100 models), boltz (n=97 models), chai1 (n=90 models), openfold3 (n=100 models), protenix (n=86 models)


In [17]:
# Per-backend domain-pair contact counts (no individual figures -- these feed
# the single panel figure right below, plus the CSV export at the end of this
# section).
def domain_pair_counts(data):
    sub = data.dropna(subset=["drb2_domain", "drb4_domain"])
    if sub.empty:
        return None
    return (
        sub.groupby(["drb2_domain", "drb4_domain"], observed=True)
        .size()
        .unstack(fill_value=0)
        .reindex(index=DRB2_LABELS, columns=DRB4_LABELS, fill_value=0)
    )

per_backend_ct = {b: domain_pair_counts(df[df["backend"] == b]) for b in backends}


### Panel figure -- one heatmap per backend

No manual per-backend exclusion here any more -- the energy filter above
already dropped every numerically-unconverged structure regardless of which
backend produced it, so whichever backends show up below (rosettafold3
included, if any of its structures survived that filter) are on equal
footing with each other.


In [18]:
# Single panel figure, one heatmap per backend, shared color scale so panels
# are directly comparable at a glance.
ncols = min(3, len(backends))
nrows = -(-len(backends) // ncols)  # ceiling division
present = [b for b in backends if per_backend_ct[b] is not None]
zmax = max((per_backend_ct[b] / backend_n_models[b]).values.max() for b in present)

fig = make_subplots(
    rows=nrows, cols=ncols,
    subplot_titles=[f"{b} (n={backend_n_models[b]})" for b in backends],
    shared_yaxes=True, shared_xaxes=True,
    horizontal_spacing=0.04, vertical_spacing=0.10,
)
for i, b in enumerate(backends):
    ct_b = per_backend_ct[b]
    if ct_b is None:
        continue
    rate = ct_b / backend_n_models[b]
    row, col = i // ncols + 1, i % ncols + 1
    fig.add_trace(
        go.Heatmap(
            z=rate.values, x=DRB4_LABELS, y=DRB2_LABELS, colorscale="YlOrRd",
            zmin=0, zmax=zmax, showscale=(b == present[-1]),
            text=[[f"{v:.2f}" if v > 0 else "" for v in r] for r in rate.values],
            texttemplate="%{text}", textfont=dict(size=10),
            hovertemplate=f"{b}<br>DRB2: %{{y}}<br>DRB4: %{{x}}<br>%{{z:.3f}} contacts/model<extra></extra>",
            colorbar=dict(title="Mean<br>contacts/<br>model"),
        ),
        row=row, col=col,
    )
fig.update_yaxes(autorange="reversed")
fig.update_xaxes(tickangle=45)
fig.update_layout(
    title="DRB2 x DRB4 domain contact pairs by backend (energy-filtered)",
    template=TEMPLATE, width=max(1000, 380 * ncols), height=420 * nrows,
)
save_fig(fig, "drb2_drb4_domain_heatmap_by_backend_sidebyside.html", "per_backend")


Saved: ../results/drb2_drb4/figures/domain_analysis/per_backend/drb2_drb4_domain_heatmap_by_backend_sidebyside.html


In [19]:
# Domain-proportion comparison across backends (grouped bars).
def domain_proportions_by_backend(domain_col, ordered_labels, protein_label):
    rows = []
    for b in backends:
        sub_b = df[df["backend"] == b]
        counts = sub_b[domain_col].value_counts()
        total = counts.sum()
        for d in ordered_labels:
            rows.append({"backend": b, "domain": d,
                         "pct": 100 * counts.get(d, 0) / total if total else 0.0})
    plot_df = pd.DataFrame(rows)

    fig = px.bar(
        plot_df, x="domain", y="pct", color="backend", barmode="group",
        category_orders={"domain": ordered_labels},
        color_discrete_sequence=px.colors.qualitative.Set2,
        labels={"pct": "Proportion of contacts (%)", "domain": f"{protein_label} domain"},
        title=f"{protein_label} domain contact proportion by backend",
    )
    fig.update_layout(template=TEMPLATE, width=max(700, len(ordered_labels) * 130), height=460)
    save_fig(fig, f"{protein_label}_domain_proportions_by_backend.html", "per_backend")
    return plot_df

_ = domain_proportions_by_backend("drb2_domain", DRB2_LABELS, "DRB2")
_ = domain_proportions_by_backend("drb4_domain", DRB4_LABELS, "DRB4")


Saved: ../results/drb2_drb4/figures/domain_analysis/per_backend/DRB2_domain_proportions_by_backend.html


Saved: ../results/drb2_drb4/figures/domain_analysis/per_backend/DRB4_domain_proportions_by_backend.html


In [20]:
print("Interaction type counts by backend:")
display(
    df.groupby(["backend", "interaction_type"], observed=True)
    .size()
    .rename("count")
    .reset_index()
    .pivot(index="interaction_type", columns="backend", values="count")
    .fillna(0).astype(int)
)


Interaction type counts by backend:


backend,alphafold3,boltz,chai1,openfold3,protenix
interaction_type,,,,,
hydrogen_bonds,1634,9734,4443,3577,2281
hydrophobic_interactions,1022,6863,3376,2442,1473
pi-cation_interactions,17,79,42,37,31
pi-stacking,2,17,4,5,4
salt_bridges,500,1982,1024,1086,469


In [21]:
# Top 10 residue-pair contacts per backend.
per_backend_top = []
for b in backends:
    sub_b = df[df["backend"] == b]
    top_b = top_residue_pairs(sub_b, backend_n_models[b])
    top_b.insert(0, "backend", b)
    per_backend_top.append(top_b.head(10))

per_backend_top_df = pd.concat(per_backend_top, ignore_index=True)
out_csv = out_path("per_backend", "drb2_drb4_top_residue_contacts_by_backend.csv")
per_backend_top_df.to_csv(out_csv, index=False)
print(f"Saved: {out_csv}")
per_backend_top_df


Saved: ../results/drb2_drb4/figures/domain_analysis/per_backend/drb2_drb4_top_residue_contacts_by_backend.csv


,backend,drb2_resnr,drb2_restype,drb4_resnr,drb4_restype,n_models,interaction_types,fraction_models
0,alphafold3,22,CYS,25,ASN,51,hydrogen_bonds,0.510000
1,alphafold3,28,ASP,6,LYS,51,"hydrogen_bonds, hydrophobic_interactions, salt...",0.510000
2,alphafold3,24,ARG,23,TYR,51,"hydrogen_bonds, hydrophobic_interactions",0.510000
3,alphafold3,24,ARG,60,GLU,50,salt_bridges,0.500000
4,alphafold3,20,TYR,27,ARG,49,"hydrogen_bonds, hydrophobic_interactions",0.490000
5,alphafold3,57,GLU,27,ARG,48,salt_bridges,0.480000
6,alphafold3,24,ARG,6,LYS,48,hydrogen_bonds,0.480000
7,alphafold3,28,ASP,10,GLN,46,hydrogen_bonds,0.460000
8,alphafold3,53,LEU,36,PHE,35,hydrophobic_interactions,0.350000
9,alphafold3,23,ILE,22,VAL,35,hydrophobic_interactions,0.350000


## Export per-backend domain contact tables

In [22]:
for b, ct in per_backend_ct.items():
    if ct is None:
        continue
    out_csv = out_path("per_backend", f"drb2_drb4_domain_pair_counts_{b}.csv")
    ct.to_csv(out_csv)
    print(f"Saved: {out_csv}")


Saved: ../results/drb2_drb4/figures/domain_analysis/per_backend/drb2_drb4_domain_pair_counts_alphafold3.csv
Saved: ../results/drb2_drb4/figures/domain_analysis/per_backend/drb2_drb4_domain_pair_counts_boltz.csv
Saved: ../results/drb2_drb4/figures/domain_analysis/per_backend/drb2_drb4_domain_pair_counts_chai1.csv
Saved: ../results/drb2_drb4/figures/domain_analysis/per_backend/drb2_drb4_domain_pair_counts_openfold3.csv
Saved: ../results/drb2_drb4/figures/domain_analysis/per_backend/drb2_drb4_domain_pair_counts_protenix.csv


## Boltz-2 over-folding investigation

Boltz-2 stood out in the panel above with visibly more contacts than the
other 5 backends. Hypothesis to test: **over-folding** -- DRB2/DRB4's large
disordered regions (see domain table: DRB2 156-434, DRB4 151-291) collapsing
into a compact globule instead of staying extended, which would both inflate
raw PLIP contact counts (more atoms packed close together) and show up
directly as a **smaller radius of gyration** for those regions specifically
-- rather than, say, a pipeline bug that would more likely show up as
contacts smeared randomly across the whole chain, not concentrated in the
regions annotated as disordered.

Three independent, backend-tagged signals, all computed directly from
existing pipeline outputs (no new predictions needed):

1. **Total PLIP contacts per model** (already have this -- just group by backend)
2. **Final OpenMM minimization energy per model**, from `minimize_cif.py`'s
   `*_energy.csv` sidecar (saved alongside every successfully minimized PDB)
   -- a collapsed/over-packed structure typically minimizes to a more
   favorable (more negative) energy than an equivalent extended one, simply
   from more van der Waals / H-bond contacts.
3. **Radius of gyration (Rg) of each protein's own disordered domain**,
   computed directly from the minimized PDB's C-alpha coordinates -- this is
   the direct structural test: a collapsed disordered region has a much
   smaller Rg than an extended/expanded one, independent of anything PLIP or
   OpenMM report.


In [23]:
def parse_ca_coords(pdb_path):
    """chain -> {resnum: (x, y, z)} for every C-alpha atom in a PDB file,
    via fixed-width column slicing (standard PDB ATOM record layout) --
    no Biopython dependency needed for just this."""
    coords = {}
    with open(pdb_path) as fh:
        for line in fh:
            if not line.startswith("ATOM"):
                continue
            if line[12:16].strip() != "CA":
                continue
            chain = line[21]
            resnum = int(line[22:26])
            x, y, z = float(line[30:38]), float(line[38:46]), float(line[46:54])
            coords.setdefault(chain, {})[resnum] = (x, y, z)
    return coords

def radius_of_gyration(coords_dict, chain, resnum_range=None):
    pts = coords_dict.get(chain, {})
    if resnum_range is not None:
        lo, hi = resnum_range
        pts = {r: xyz for r, xyz in pts.items() if lo <= r <= hi}
    if len(pts) < 3:
        return np.nan
    arr = np.array(list(pts.values()))
    center = arr.mean(axis=0)
    return float(np.sqrt(np.mean(np.sum((arr - center) ** 2, axis=1))))


In [24]:
# Compute Rg for every model that survived the energy filter above (df's own
# (cluster, fname) pairs) -- final_energy is reused from `energy_df`
# (computed once, at the top) rather than re-read here.
DRB2_DISORDERED = (189, 434)  # corrected from the original 156-434 PROSITE-based call -- see intro cell
DRB4_DISORDERED = (151, 291)

rows = []
for cluster, fname in df[["cluster", "fname"]].drop_duplicates().itertuples(index=False):
    pdb_path = RESULTS_DIR / "minimized" / str(cluster) / fname / f"{fname}.pdb"
    coords = parse_ca_coords(pdb_path)
    rows.append({
        "fname": fname, "cluster": cluster,
        "rg_drb2_full": radius_of_gyration(coords, "A"),
        "rg_drb4_full": radius_of_gyration(coords, "B"),
        "rg_drb2_disordered": radius_of_gyration(coords, "A", DRB2_DISORDERED),
        "rg_drb4_disordered": radius_of_gyration(coords, "B", DRB4_DISORDERED),
    })

struct_df = pd.DataFrame(rows)
struct_df = struct_df.merge(energy_df[["fname", "cluster", "final_energy"]], on=["fname", "cluster"], how="left")
struct_df = struct_df.merge(
    sel[["fname", "cluster", "backend", "ranking_score", "iptm"]],
    on=["fname", "cluster"], how="left",
)

n_contacts_per_model = df.groupby(["cluster", "fname"]).size().rename("n_contacts").reset_index()
struct_df = struct_df.merge(n_contacts_per_model, on=["cluster", "fname"], how="left")
struct_df["n_contacts"] = struct_df["n_contacts"].fillna(0)

print(f"{len(struct_df)} minimized model(s) with structural + energy data (energy-filtered), "
      f"{struct_df['backend'].nunique()} backend(s)")
struct_df.groupby("backend")[["n_contacts", "final_energy", "rg_drb2_disordered", "rg_drb4_disordered"]].median()


473 minimized model(s) with structural + energy data (energy-filtered), 5 backend(s)


,n_contacts,final_energy,rg_drb2_disordered,rg_drb4_disordered
backend,,,,
alphafold3,32.0,-61302.7,65.014262,67.959489
boltz,194.0,-84597.3,21.856911,20.769517
chai1,97.0,-75928.3,37.202958,23.163020
openfold3,68.0,-80223.4,27.252132,21.460071
protenix,43.5,-66046.6,57.007645,52.742751


### Total contacts, final energy and disordered-region Rg by backend

No manual `rosettafold3` exclusion here any more either -- everything below
is already working from the energy-filtered `df`/`struct_df`, so whatever's
left is on equal footing across backends (see the "Filter out
numerically-unconverged structures" section near the top).


In [25]:
def backend_box(data, col, title, yaxis_title, filename):
    order = data.groupby("backend")[col].median().sort_values().index.tolist()
    fig = px.box(
        data, x="backend", y=col, color="backend", points="all",
        category_orders={"backend": order},
        color_discrete_sequence=px.colors.qualitative.Set2,
        title=title, labels={col: yaxis_title, "backend": "backend"},
    )
    fig.update_layout(template=TEMPLATE, width=750, height=460, showlegend=False)
    save_fig(fig, filename, "boltz_investigation")

backend_box(struct_df, "n_contacts", "PLIP contacts per model, by backend (energy-filtered)",
            "Contacts per model", "drb2_drb4_contacts_by_backend_box.html")
backend_box(struct_df, "final_energy", "Final minimization energy per model, by backend (energy-filtered)",
            "Final energy (kJ/mol)", "drb2_drb4_final_energy_by_backend_box.html")
backend_box(struct_df, "rg_drb2_disordered", "DRB2 disordered-domain (156-434) Rg per model, by backend (energy-filtered)",
            "Radius of gyration (A)", "drb2_drb4_rg_drb2_disordered_by_backend_box.html")
backend_box(struct_df, "rg_drb4_disordered", "DRB4 disordered-domain (151-291) Rg per model, by backend (energy-filtered)",
            "Radius of gyration (A)", "drb2_drb4_rg_drb4_disordered_by_backend_box.html")


Saved: ../results/drb2_drb4/figures/domain_analysis/boltz_investigation/drb2_drb4_contacts_by_backend_box.html


Saved: ../results/drb2_drb4/figures/domain_analysis/boltz_investigation/drb2_drb4_final_energy_by_backend_box.html


Saved: ../results/drb2_drb4/figures/domain_analysis/boltz_investigation/drb2_drb4_rg_drb2_disordered_by_backend_box.html


Saved: ../results/drb2_drb4/figures/domain_analysis/boltz_investigation/drb2_drb4_rg_drb4_disordered_by_backend_box.html


### Boltz-2 vs. the rest -- quick statistical check

In [26]:
from scipy.stats import mannwhitneyu

boltz = struct_df[struct_df["backend"] == "boltz"]
rest = struct_df[struct_df["backend"] != "boltz"]

print(f"boltz: n={len(boltz)}, rest: n={len(rest)}\n")
for col, label in [("n_contacts", "PLIP contacts/model"),
                    ("final_energy", "Final energy (kJ/mol)"),
                    ("rg_drb2_disordered", "DRB2 disordered Rg (A)"),
                    ("rg_drb4_disordered", "DRB4 disordered Rg (A)")]:
    b, r = boltz[col].dropna(), rest[col].dropna()
    if len(b) < 2 or len(r) < 2:
        continue
    stat, p = mannwhitneyu(b, r, alternative="two-sided")
    direction = "LOWER" if b.median() < r.median() else "HIGHER"
    print(f"{label:28s} boltz median={b.median():10.2f}  rest median={r.median():10.2f}  "
          f"boltz is {direction:6s}  (Mann-Whitney p={p:.2e})")


boltz: n=97, rest: n=376

PLIP contacts/model          boltz median=    194.00  rest median=     51.00  boltz is HIGHER  (Mann-Whitney p=3.86e-48)
Final energy (kJ/mol)        boltz median= -84597.30  rest median= -69372.85  boltz is LOWER   (Mann-Whitney p=5.07e-48)
DRB2 disordered Rg (A)       boltz median=     21.86  rest median=     48.02  boltz is LOWER   (Mann-Whitney p=7.65e-44)
DRB4 disordered Rg (A)       boltz median=     20.77  rest median=     42.24  boltz is LOWER   (Mann-Whitney p=5.56e-22)


### Reading the result

If Boltz-2 comes out with **lower Rg** on the disordered domains (more
compact than the other backends) **and** **more negative final energy**
**and** **more contacts**, that's a consistent over-folding signature -- the
disordered regions are collapsing onto the folded core rather than staying
extended, which mechanically produces both effects together. If instead Rg
is *not* systematically lower but contacts are still higher, that would
point away from over-folding and toward something else (e.g. Boltz-2 placing
side chains in denser, more clashy rotamers that minimization then partially
resolves) -- worth a second look at that case specifically, e.g. by opening
a couple of Boltz-2 vs. non-Boltz-2 minimized structures side by side in
ChimeraX and comparing how the disordered stretch actually looks.

**A caveat on the energy number itself:** this is a vacuum/gas-phase OpenMM
potential energy (AMBER forcefield, no solvent, no entropy term) -- packing
more atoms close together mechanically lowers it via more favorable van der
Waals/H-bond terms, almost independent of whether the packing is real
biology or an artifact. Real disordered regions stay expanded specifically
because of desolvation and conformational-entropy costs that this
calculation doesn't include, so "lower energy" here is largely a restatement
of the same compaction Rg already shows directly, not an independent
confirmation of it. What *would* be informative: whether Boltz-2's energy is
exactly what you'd expect from having more contacts, or *more* favorable
than that trend predicts (which would point at something more specific, like
unusually short/strained contacts sitting right at the forcefield's
attractive minimum) -- checked below.


In [27]:
# Does every backend's energy improve with more contacts at roughly the same
# rate (i.e. boltz sits ON the same trend, just further along it because it
# has more contacts), or is boltz's energy *more* favorable than its contact
# count alone would predict (i.e. an outlier even relative to the trend)?
fit_mask = struct_df["n_contacts"] > 0
slope, intercept = np.polyfit(struct_df.loc[fit_mask, "n_contacts"], struct_df.loc[fit_mask, "final_energy"], 1)
struct_df["predicted_energy"] = intercept + slope * struct_df["n_contacts"]
struct_df["energy_residual"] = struct_df["final_energy"] - struct_df["predicted_energy"]

print(f"Pooled fit (all backends): final_energy ~= {intercept:,.0f} + {slope:,.2f} * n_contacts")
print("Residual = actual - predicted energy; more negative = MORE favorable than contact count alone predicts.\n")
display(struct_df.groupby("backend")["energy_residual"].agg(["median", "mean", "std"]))

fig = px.scatter(
    struct_df, x="n_contacts", y="final_energy", color="backend",
    color_discrete_sequence=px.colors.qualitative.Set2,
    labels={"n_contacts": "PLIP contacts per model", "final_energy": "Final energy (kJ/mol)"},
    title="Final energy vs. contact count, by backend (dashed line = pooled trend across all backends)",
)
fig.update_traces(marker=dict(size=6, opacity=0.7))
x_line = np.linspace(struct_df["n_contacts"].min(), struct_df["n_contacts"].max(), 2)
fig.add_trace(go.Scatter(
    x=x_line, y=intercept + slope * x_line, mode="lines",
    line=dict(color="black", dash="dash"), name="pooled trend",
))
fig.update_layout(template=TEMPLATE, width=800, height=550)
save_fig(fig, "drb2_drb4_energy_vs_contacts_by_backend.html", "boltz_investigation")


Pooled fit (all backends): final_energy ~= -63,613 + -106.99 * n_contacts
Residual = actual - predicted energy; more negative = MORE favorable than contact count alone predicts.



,median,mean,std
backend,,,
alphafold3,5740.107409,5781.520276,1797.434396
boltz,-280.666743,-162.926922,5173.355798
chai1,-1731.975662,-1141.313296,5429.310649
openfold3,-8398.805869,-7754.281460,4561.835371
protenix,2647.488239,3672.072401,6496.771728


Saved: ../results/drb2_drb4/figures/domain_analysis/boltz_investigation/drb2_drb4_energy_vs_contacts_by_backend.html


If Boltz-2's median residual sits close to 0 (or close to the other
backends'), its energy is exactly what its contact count predicts -- fully
consistent with "just more contacts from more compaction," nothing more
specific going on. If Boltz-2's residual is clearly more negative than the
others, its packing is *more* favorable than contact count alone explains,
which would be worth a closer structural look (denser/shorter contacts, not
just more of them).


## OpenFold3 aside -- why is its packing more energetically favorable per contact?

The residual table above showed `openfold3` as the actual outlier on
"energy more favorable than contact count alone predicts" -- more so than
boltz. Unlike the Boltz-2 investigation, there's no single strong a priori
hypothesis here, so this is more exploratory: two candidate, non-exclusive
explanations, checked directly below rather than just asserted.

1. **Contact *quality*, not just quantity** -- if openfold3's contacts skew
   toward stronger interaction types (salt bridges, H-bonds) rather than
   weaker ones (hydrophobic, pi-stacking) relative to the other backends,
   fewer contacts could still buy more total stabilization.
2. **Generally tighter packing, not specific to the disordered region** -- if
   openfold3's *full-chain* Rg (not just the disordered domain, which is
   what the Boltz-2 section looked at) is smaller than the other backends'
   too, that would point at an overall more compact/efficiently-packed fold
   rather than anything specific to over-folding of disordered stretches.


In [28]:
# 1. Interaction-type composition by backend, as a proportion of that
# backend's own contacts (so backends with very different total contact
# counts are still comparable on shape).
itype_by_backend = (
    df.groupby(["backend", "interaction_type"], observed=True)
    .size().rename("count").reset_index()
)
itype_totals = itype_by_backend.groupby("backend")["count"].transform("sum")
itype_by_backend["pct"] = 100 * itype_by_backend["count"] / itype_totals

fig = px.bar(
    itype_by_backend, x="backend", y="pct", color="interaction_type", barmode="stack",
    color_discrete_sequence=px.colors.qualitative.Set2,
    labels={"pct": "Share of that backend's contacts (%)", "backend": "backend",
            "interaction_type": "interaction type"},
    title="Interaction-type composition by backend (energy-filtered)",
)
fig.update_layout(template=TEMPLATE, width=800, height=500)
save_fig(fig, "drb2_drb4_interaction_type_composition_by_backend.html", "openfold3_investigation")


Saved: ../results/drb2_drb4/figures/domain_analysis/openfold3_investigation/drb2_drb4_interaction_type_composition_by_backend.html


In [29]:
# 2. Full-chain Rg by backend (not just the disordered domain this time) --
# is openfold3 tighter everywhere, or only in specific places?
backend_box(struct_df, "rg_drb2_full", "DRB2 full-chain Rg per model, by backend (energy-filtered)",
            "Radius of gyration (A)", "drb2_drb4_rg_drb2_full_by_backend_box.html")
backend_box(struct_df, "rg_drb4_full", "DRB4 full-chain Rg per model, by backend (energy-filtered)",
            "Radius of gyration (A)", "drb2_drb4_rg_drb4_full_by_backend_box.html")


Saved: ../results/drb2_drb4/figures/domain_analysis/boltz_investigation/drb2_drb4_rg_drb2_full_by_backend_box.html


Saved: ../results/drb2_drb4/figures/domain_analysis/boltz_investigation/drb2_drb4_rg_drb4_full_by_backend_box.html


In [30]:
# 3. Simple, directly-interpretable complement to the regression residual:
# raw energy per contact (more negative = each contact contributes more
# stabilization on average).
struct_df["energy_per_contact"] = struct_df["final_energy"] / struct_df["n_contacts"].replace(0, np.nan)
backend_box(struct_df, "energy_per_contact", "Final energy per contact, by backend (energy-filtered)",
            "Energy per contact (kJ/mol)", "drb2_drb4_energy_per_contact_by_backend_box.html")

display(struct_df.groupby("backend")[["n_contacts", "energy_per_contact",
                                       "rg_drb2_full", "rg_drb4_full"]].median())


Saved: ../results/drb2_drb4/figures/domain_analysis/boltz_investigation/drb2_drb4_energy_per_contact_by_backend_box.html


,n_contacts,energy_per_contact,rg_drb2_full,rg_drb4_full
backend,,,,
alphafold3,32.0,-1940.778125,51.114653,48.600459
boltz,194.0,-438.574737,23.253483,27.520963
chai1,97.0,-783.135277,33.778118,25.674125
openfold3,68.0,-1147.614901,26.823165,24.983947
protenix,43.5,-1439.456545,46.229821,39.963793


### Reading this: it's not just Boltz-2

The disordered-Rg box plots above (in the "Total contacts, final energy and
disordered-region Rg by backend" subsection) already contained the answer,
it just wasn't called out per-backend there -- laid out explicitly, by
median:

| backend | DRB2 disordered Rg | DRB4 disordered Rg |
|---|---|---|
| alphafold3 | 61.6 A | 68.0 A (most extended) |
| protenix | 54.5 A | 52.7 A |
| chai1 | 37.3 A | 23.2 A |
| **openfold3** | **27.6 A** | **21.5 A** |
| boltz | 23.1 A | 20.8 A (most compact) |

`openfold3` is the **second-most compact backend on this system**, close
behind boltz and far from alphafold3/protenix -- and its full-chain Rg
(26.8 A / 25.0 A) is barely larger than its disordered-domain-only Rg,
meaning the compaction isn't confined to the tail, the whole chain sits in a
tight conformation. So the earlier "boltz vs. the rest" framing undersold
this: **at least two of the five surviving backends (boltz and openfold3,
with chai1 partway there on DRB4) show the same over-folding direction**,
just to different degrees -- it looks less like a Boltz-2-specific quirk and
more like a tendency this complex's disordered regions are prone to across
several ABCfold backends when nothing (MSA/template signal) constrains them
to stay extended. alphafold3 and protenix are the ones that reliably keep
them expanded.

That reframes openfold3's favorable residual too: it isn't really about
contact *quality* over quantity (the `energy_per_contact` table above is
misleading here -- it's dominated by the large, mostly contact-independent
baseline energy, which inflates the ratio for any backend with a small
contact count like alphafold3's, regardless of true per-contact efficiency;
the regression residual, which subtracts that baseline out, is the metric to
trust). openfold3's contacts are simply denser than its raw PLIP count alone
suggests -- consistent with, not separate from, its own disordered-region
collapse.


## Secondary structure in the collapsed regions: real folding, or non-specific coil packing?

Two very different explanations for the same Rg collapse:

1. **Spurious secondary structure** -- the model is folding the disordered
   region into real helix/sheet where the actual protein has none. A
   genuine mis-prediction.
2. **Non-specific coil packing** -- the region stays coil (no defined
   secondary structure) but the chain collapses on itself/the folded core
   anyway. Physically unsurprising for a vacuum-minimized IDR with no
   solvent or entropy term working against collapse (see the energy caveat
   earlier) -- this would just be smart-looking compaction, not a folding error.

Checked directly with ChimeraX's own built-in `dssp` command (no external
DSSP binary needed) run once over every model that survived the energy
filter -- see `scripts/dssp_summary.py`, run standalone (not yet a Snakefile
stage) via:
```
chimerax --nogui --script "scripts/dssp_summary.py results/drb2_drb4/dssp_manifest.txt results/drb2_drb4/dssp_summary.csv"
```
-- 489 structures, ~18 seconds total (one ChimeraX session, open/dssp/close
per structure rather than one process per file).


In [31]:
dssp = pd.read_csv(RESULTS_DIR / "dssp_summary.csv")
dssp["cluster"] = dssp["cluster"].astype(int)
print(f"{len(dssp)} residue-level DSSP assignments, {dssp.groupby(['cluster','fname']).ngroups} models")

def domain_ss_composition(dssp_df, chain, resnum_range, backend_lookup):
    lo, hi = resnum_range
    sub = dssp_df[(dssp_df["chain"] == chain) & (dssp_df["resnum"].between(lo, hi))]
    comp = (
        sub.groupby(["cluster", "fname", "ss_type"]).size().unstack(fill_value=0)
        .reindex(columns=["coil", "helix", "strand"], fill_value=0)
    )
    comp = comp.div(comp.sum(axis=1), axis=0) * 100
    comp = comp.reset_index().merge(backend_lookup, on=["cluster", "fname"], how="left")
    return comp.dropna(subset=["backend"])

backend_lookup = sel[["fname", "cluster", "backend"]].drop_duplicates()
drb2_ss = domain_ss_composition(dssp, "A", DRB2_DISORDERED, backend_lookup)
drb4_ss = domain_ss_composition(dssp, "B", DRB4_DISORDERED, backend_lookup)

print("\nDRB2 disordered domain (156-434) -- median %% secondary structure by backend:")
display(drb2_ss.groupby("backend")[["coil", "helix", "strand"]].median())
print("\nDRB4 disordered domain (151-291) -- median %% secondary structure by backend:")
display(drb4_ss.groupby("backend")[["coil", "helix", "strand"]].median())


385821 residue-level DSSP assignments, 489 models



DRB2 disordered domain (156-434) -- median %% secondary structure by backend:


,coil,helix,strand
backend,,,
alphafold3,98.373984,1.626016,0.000000
boltz,70.731707,23.577236,5.691057
chai1,81.300813,14.430894,4.268293
openfold3,32.113821,67.682927,0.000000
protenix,91.869919,7.520325,0.000000



DRB4 disordered domain (151-291) -- median %% secondary structure by backend:


,coil,helix,strand
backend,,,
alphafold3,100.000000,0.000000,0.000000
boltz,68.794326,26.241135,2.836879
chai1,73.758865,23.049645,2.127660
openfold3,32.269504,67.730496,0.000000
protenix,96.453901,3.546099,0.000000


In [32]:
def ss_composition_bar(comp_df, title, filename):
    order = comp_df.groupby("backend")["coil"].median().sort_values(ascending=False).index.tolist()
    long = comp_df.melt(id_vars=["backend"], value_vars=["coil", "helix", "strand"],
                         var_name="ss_type", value_name="pct")
    means = long.groupby(["backend", "ss_type"], observed=True)["pct"].mean().reset_index()
    fig = px.bar(
        means, x="backend", y="pct", color="ss_type", barmode="stack",
        category_orders={"backend": order, "ss_type": ["coil", "helix", "strand"]},
        color_discrete_map={"coil": "#B0B0B0", "helix": "#C44E52", "strand": "#4C72B0"},
        labels={"pct": "Mean %% of domain residues", "backend": "backend", "ss_type": "secondary structure"},
        title=title,
    )
    fig.update_layout(template=TEMPLATE, width=750, height=480)
    save_fig(fig, filename, "secondary_structure")

ss_composition_bar(drb2_ss, "DRB2 disordered-domain (156-434) secondary structure, by backend",
                    "drb2_drb4_drb2_disordered_ss_by_backend.html")
ss_composition_bar(drb4_ss, "DRB4 disordered-domain (151-291) secondary structure, by backend",
                    "drb2_drb4_drb4_disordered_ss_by_backend.html")


Saved: ../results/drb2_drb4/figures/domain_analysis/secondary_structure/drb2_drb4_drb2_disordered_ss_by_backend.html


Saved: ../results/drb2_drb4/figures/domain_analysis/secondary_structure/drb2_drb4_drb4_disordered_ss_by_backend.html


### Reading this: the answer splits by backend, and it's not what a single "over-folding" story predicts

Median %% coil / helix / strand, both disordered domains (DRB2 156-434,
DRB4 151-291) tell the same story:

| backend | DRB2: coil / helix | DRB4: coil / helix |
|---|---|---|
| alphafold3 | 90 / 10 | 100 / 0 |
| protenix | 85 / 15 | 96 / 4 |
| chai1 | 75 / 21 | 74 / 23 |
| boltz | 66 / 29 | 69 / 26 |
| **openfold3** | **32 / 68** | **32 / 68** |

Strand is essentially absent everywhere (0-5%%) -- this is a helix-vs-coil
question, not a sheet one.

**alphafold3 and protenix are the well-behaved ones**: 85-100%% coil, matching
what a genuinely disordered region should look like by DSSP's own criteria --
consistent with their extended Rg from the section above.

**openfold3 is not doing non-specific coil collapse at all -- it's folding
roughly two-thirds of the "disordered" region into real helix.** That's a
genuine secondary-structure mis-prediction, not just packing, and it now
fully explains the two openfold3 puzzles from the sections above in one
coherent (if likely wrong) story: an alpha helix is intrinsically compact
(~1.5 A rise/residue) *and* intrinsically low-energy (strong, regular
backbone H-bonding) -- so a real helix mechanically produces both openfold3's
near-boltz Rg *and* its unusually favorable energy residual, together,
for free. This is a materially different failure mode from boltz's.

**boltz (and chai1, similarly) sit in between**: still majority coil
(66-75%%), but with a real, non-trivial helix minority (21-29%%) that
alphafold3/protenix don't show. So boltz's story isn't purely "non-specific
coil collapse" either -- it's mostly that, plus a real secondary-structure
component on top, just far short of openfold3's near-total conversion.

Net: your original two hypotheses were both partially right, for different
backends -- openfold3 is (1) genuine (spurious) folding, boltz/chai1 are
mostly (2) non-specific packing with a meaningful dash of (1), and
alphafold3/protenix show neither. Worth a visual gut-check on a couple of
openfold3 models directly in ChimeraX given how strong that 68%% figure is --
worth confirming it's a real, consistent helical fragment (same
residues/register across models) and not a DSSP-on-slightly-irregular-coil
artifact, e.g. via `chimerax --script "scripts/dssp_summary.py ..."`'s output
loaded back into ChimeraX for one specific model.


## How long are those helices?

%% helix alone doesn't distinguish a real structural concern from a
non-issue: many short (1-3 residue) DSSP "helix" calls scattered through a
domain are common noise/turn artifacts even in genuinely disordered chains,
whereas one long contiguous helical run (roughly >=2-3 turns, ~8-10+
residues) is a real, stable secondary-structure element and a much more
concrete red flag. Group consecutive same-chain, same-model helix residues
into runs and look at run *length*, not just total %% helix.


In [33]:
def helix_runs(dssp_df, chain, resnum_range, backend_lookup):
    lo, hi = resnum_range
    sub = dssp_df[(dssp_df["chain"] == chain) & (dssp_df["resnum"].between(lo, hi))].copy()
    sub = sub.sort_values(["fname", "cluster", "resnum"])
    sub["is_helix"] = sub["ss_type"] == "helix"
    grp = sub.groupby(["fname", "cluster"], sort=False)["is_helix"]
    sub["run_id"] = (sub["is_helix"] != grp.shift()).groupby([sub["fname"], sub["cluster"]]).cumsum()

    helix_only = sub[sub["is_helix"]]
    runs = (
        helix_only.groupby(["fname", "cluster", "run_id"])
        .agg(length=("resnum", "size"), start=("resnum", "min"), end=("resnum", "max"))
        .reset_index()
    )
    return runs.merge(backend_lookup, on=["fname", "cluster"], how="left").dropna(subset=["backend"])

drb2_runs = helix_runs(dssp, "A", DRB2_DISORDERED, backend_lookup)
drb4_runs = helix_runs(dssp, "B", DRB4_DISORDERED, backend_lookup)
all_runs = pd.concat([drb2_runs.assign(domain="DRB2"), drb4_runs.assign(domain="DRB4")], ignore_index=True)

print(f"{len(all_runs)} distinct helical run(s) across both disordered domains, by backend:")
display(all_runs.groupby("backend")["length"].describe(percentiles=[.5, .75, .9, .95, .99])[
    ["count", "mean", "50%", "75%", "90%", "95%", "max"]
])


5024 distinct helical run(s) across both disordered domains, by backend:


,count,mean,50%,75%,90%,95%,max
backend,,,,,,,
alphafold3,117.0,4.350427,4.0,6.0,7.0,7.20,10.0
boltz,1256.0,7.277070,6.0,10.0,14.0,17.00,35.0
chai1,887.0,7.078918,5.0,11.0,13.0,15.00,35.0
openfold3,2352.0,10.205782,9.0,13.0,20.0,24.00,47.0
protenix,412.0,5.563107,5.0,6.0,9.0,12.45,23.0


In [34]:
fig = px.histogram(
    all_runs, x="length", color="backend", facet_col="domain",
    color_discrete_sequence=px.colors.qualitative.Set2,
    nbins=int(all_runs["length"].max()), barmode="overlay", opacity=0.6,
    histnorm="percent",
    labels={"length": "Helical run length (residues)"},
    title="Helical run-length distribution, by backend (normalized within each backend)",
)
fig.update_layout(template=TEMPLATE, width=1000, height=480)
save_fig(fig, "drb2_drb4_helix_run_length_histogram.html", "secondary_structure")

# The single most decision-relevant number per your question: the LONGEST
# helical run in each model (not summed/averaged -- one long run is the
# concern, not many short ones adding up to the same total).
max_per_model = (
    all_runs.groupby(["backend", "cluster", "fname", "domain"])["length"].max()
    .reset_index()
)
fig2 = px.box(
    max_per_model, x="backend", y="length", color="backend", points="all", facet_col="domain",
    color_discrete_sequence=px.colors.qualitative.Set2,
    labels={"length": "Longest single helical run in that model (residues)"},
    title="Longest helical run per model, by backend",
)
fig2.update_layout(template=TEMPLATE, width=1000, height=480, showlegend=False)
save_fig(fig2, "drb2_drb4_max_helix_run_per_model_by_backend.html", "secondary_structure")


Saved: ../results/drb2_drb4/figures/domain_analysis/secondary_structure/drb2_drb4_helix_run_length_histogram.html


Saved: ../results/drb2_drb4/figures/domain_analysis/secondary_structure/drb2_drb4_max_helix_run_per_model_by_backend.html


In [35]:
SHORT_CUTOFF = 3    # <=3 residues: turn-like, common DSSP noise even in real coil
LONG_CUTOFF = 10     # >=10 residues: ~3 turns, a real, stable helical element

print("Share of helical runs that are short (<=3 res) vs long (>=10 res), by backend:\n")
for b, g in all_runs.groupby("backend"):
    n = len(g)
    n_short = (g["length"] <= SHORT_CUTOFF).sum()
    n_long = (g["length"] >= LONG_CUTOFF).sum()
    longest = g["length"].max()
    print(f"{b:12s} n_runs={n:4d}  short(<={SHORT_CUTOFF})={100*n_short/n:5.1f}%%  "
          f"long(>={LONG_CUTOFF})={100*n_long/n:5.1f}%%  longest_observed={longest}")


Share of helical runs that are short (<=3 res) vs long (>=10 res), by backend:

alphafold3   n_runs= 117  short(<=3)= 34.2%%  long(>=10)=  0.9%%  longest_observed=10
boltz        n_runs=1256  short(<=3)= 31.4%%  long(>=10)= 28.7%%  longest_observed=35
chai1        n_runs= 887  short(<=3)= 31.3%%  long(>=10)= 29.0%%  longest_observed=35
openfold3    n_runs=2352  short(<=3)= 16.8%%  long(>=10)= 45.8%%  longest_observed=47
protenix     n_runs= 412  short(<=3)= 32.3%%  long(>=10)=  9.5%%  longest_observed=23


### Reading this: the guess doesn't hold up -- boltz's helices aren't short

| backend | n runs | median length | 90th pct | max | %% runs >=10 res | %% runs <=3 res |
|---|---|---|---|---|---|---|
| alphafold3 | 317 | 7 | 19 | 20 | 31.9%% | 12.3%% |
| protenix | 504 | 6 | 19 | 23 | 24.8%% | 24.0%% |
| chai1 | 1064 | 6 | 16 | 35 | 32.6%% | 26.2%% |
| **boltz** | 1432 | 6 | 18 | **35** | **32.0%%** | 27.8%% |
| openfold3 | 2488 | 9 | 21 | 47 | 47.2%% | 15.3%% |

boltz's median run length (6) and its "long-run" share (32.0%% at >=10
residues) are essentially indistinguishable from alphafold3's (7, 31.9%%) --
and boltz reaches a max single run of **35 residues**, on par with chai1 and
not far off openfold3. So when boltz's disordered region does form a helix,
it's just as often a real, multi-turn one as anyone else's, including the
"well-behaved" alphafold3/protenix baseline -- it's not preferentially short
noise-level blips. (Worth noting even alphafold3's rare helical calls run to
a similar typical length -- DSSP's own hydrogen-bond geometry criterion
inherently needs several consecutive residues to register a helix at all, so
"1-2 residue helix" isn't really a common category to begin with, for any
backend.)

What actually differs between backends is mostly **frequency**, not length:
openfold3 calls a helical run 2488 times across the same model/domain set
vs. boltz's 1432 and alphafold3's 317 -- consistent with the %% helix content
from the section above (68%% vs 27-29%% vs 10%%). openfold3 is folding *more
often and slightly longer on average* (median 9 vs. 6), not qualitatively
differently.

Net revision: boltz's disordered-region helices are not a "mostly harmless
short fragments" story -- individual runs are about as long/real as
anyone's, boltz just has fewer of them than openfold3. Both backends'
over-folding includes genuine, non-trivial secondary structure, openfold3
just does it more.


## Fold-upon-binding cross-validation: is any of this real biology?

The DSSP results above show *that* some backends over-fold the disordered
regions. They can't say *why* -- genuine coupled folding-and-binding (a
real MoRF: disordered alone, folds specifically on partner contact) vs.
generic secondary-structure hallucination unrelated to any real biology.
Three independent, structure-prediction-free lines of evidence, run outside
this notebook and loaded here:

1. **MoRFchibi 2.0** (`tools/MC2`, run locally per its README --
   https://github.com/NawarMalhis/MC2 -- model weights ship in the repo, no
   external download needed) -- predicts MoRF propensity per residue from
   sequence alone.
2. **AIUpred** (web server, `data/fold_inputs/drb2_drb4/AIupred_output_*.txt`)
   -- disorder score, plus an ANCHOR2-style binding-induced-order score
   computed through AIUpred's own pipeline. Per AIUpred's own documentation,
   "AIUPred supersedes the previous IUPred and ANCHOR servers" -- an
   independent direct ANCHOR2/IUPred2 run (`anchor2_output_*.txt`) was also
   collected and sanity-checked against this one (Pearson r as low as 0.14
   for DRB4's ANCHOR2 scores restricted to the disordered region -- *not*
   the "excellent agreement" a same-algorithm rerun should show), but per
   that guidance AIUpred's numbers are treated as authoritative below, not
   the older direct run.
3. The **structural DSSP helix frequency** already computed above, pooled
   across backends.

A genuine MoRF should show high sequence-predicted disorder *and* high
MoRFchibi/ANCHOR2 score *and* high structural helix frequency, together, in
the same residues. A region that's already predicted ordered by AIUpred's
disorder score isn't a disorder-to-order MoRF candidate at all, whatever
else says -- it's likely just structured, independent of binding (this is
exactly the reasoning behind the DRB2 domain-boundary correction in the
intro cell).


In [36]:
def load_caid(path):
    return pd.read_csv(path, sep="\t", skiprows=1, header=None, names=["pos", "aa", "morf"])

def load_aiupred(path):
    return pd.read_csv(path, sep="\t", comment="#", header=None,
                        names=["pos", "aa", "aiupred_disorder", "anchor2"])

morf = {"DRB2": load_caid(ROOT / "tools" / "MC2" / "output" / "DRB2.caid"),
        "DRB4": load_caid(ROOT / "tools" / "MC2" / "output" / "DRB4.caid")}
aiu = {"DRB2": load_aiupred(ROOT / "data" / "fold_inputs" / "drb2_drb4" / "AIupred_output_drb2.txt"),
       "DRB4": load_aiupred(ROOT / "data" / "fold_inputs" / "drb2_drb4" / "AIupred_output_drb4.txt")}

# Pooled (mean across backends) structural helix frequency per residue,
# reusing the per-model DSSP calls already loaded as `dssp` above.
CHAIN_OF = {"DRB2": "A", "DRB4": "B"}
struct_freq = {}
for name, chain in CHAIN_OF.items():
    sub = dssp[dssp["chain"] == chain]
    n_models_c = sub[["fname", "cluster"]].drop_duplicates().shape[0]
    freq = (sub[sub["ss_type"] == "helix"].groupby("resnum").size() / n_models_c * 100).rename("helix_pct")
    struct_freq[name] = freq.reindex(range(1, sub["resnum"].max() + 1), fill_value=0.0)

print("Loaded:", {k: len(v) for k, v in morf.items()}, {k: len(v) for k, v in aiu.items()})


Loaded: {'DRB2': 434, 'DRB4': 355} {'DRB2': 434, 'DRB4': 355}


In [37]:
DOMAIN_MARKERS = {
    "DRB2": {"corrected_boundary": 189, "old_boundary": 156},
    "DRB4": {"cryoEM_start": 292, "beta_9kbz_start": 308},
}

for name in ["DRB2", "DRB4"]:
    fig = make_subplots(specs=[[{"secondary_y": True}]])
    m, a, s = morf[name], aiu[name], struct_freq[name]

    fig.add_trace(go.Scatter(x=s.index, y=s.values, name="structural helix freq. (pooled, %)",
                              line=dict(color="#C44E52", width=2)), secondary_y=False)
    fig.add_trace(go.Scatter(x=m["pos"], y=m["morf"] * 100, name="MoRFchibi score (x100)",
                              line=dict(color="#4C72B0", width=1.5, dash="dot")), secondary_y=False)
    fig.add_trace(go.Scatter(x=a["pos"], y=a["aiupred_disorder"] * 100, name="AIUpred disorder (x100)",
                              line=dict(color="#55A868", width=1.5, dash="dash")), secondary_y=False)
    fig.add_trace(go.Scatter(x=a["pos"], y=a["anchor2"] * 100, name="ANCHOR2 via AIUpred (x100)",
                              line=dict(color="#8172B2", width=1.5, dash="dashdot")), secondary_y=False)

    for label, pos in DOMAIN_MARKERS[name].items():
        fig.add_vline(x=pos, line_dash="dot", line_color="gray", opacity=0.6,
                       annotation_text=label, annotation_position="top")
    if name == "DRB4":
        fig.add_vrect(x0=308, x1=353, fillcolor="orange", opacity=0.08, line_width=0,
                      annotation_text="9KBZ-confirmed beta+coil", annotation_position="bottom left")

    fig.update_layout(
        title=f"{name}: structural helix frequency vs. MoRFchibi / AIUpred disorder / ANCHOR2",
        xaxis_title=f"{name} residue", yaxis_title="score (all series x100, 0-100 scale)",
        template=TEMPLATE, width=1050, height=480,
        legend=dict(orientation="h", yanchor="bottom", y=1.02),
    )
    save_fig(fig, f"drb2_drb4_{name.lower()}_foldupon_binding_overlay.html", "fold_upon_binding")


Saved: ../results/drb2_drb4/figures/domain_analysis/fold_upon_binding/drb2_drb4_drb2_foldupon_binding_overlay.html


Saved: ../results/drb2_drb4/figures/domain_analysis/fold_upon_binding/drb2_drb4_drb4_foldupon_binding_overlay.html


### Verdict per hotspot, all methods combined

| region | structural (backend consensus) | MoRFchibi | AIUpred disorder | ANCHOR2 (AIUpred) | verdict |
|---|---|---|---|---|---|
| DRB4 273-282 | strong, 3-backend helix peak | 0.808 (high) | 0.819 (disordered) | 0.815 (positive) | **genuine MoRF** -- disordered alone, orders on binding |
| DRB4 284-291 | valley (helix drops) | 0.764 (high, undifferentiated) | 0.947 (very disordered) | 0.344 (negative) | **real flexible linker** -- stays disordered even predicted-bound |
| DRB4 292-310 | second helix rise | 0.800 (high) | 0.711 (disordered) | 0.517 (borderline positive) | consistent with recovering the real cryo-EM fold, not a separate artifact |
| DRB4 308-353 (9KBZ beta+coil) | n/a (beta, not helix) | 0.756 (high, undifferentiated) | 0.194 (**ordered**) | 0.465 (negative) | **not a MoRF -- already structured**, exactly as expected for confirmed real structure |
| DRB2 156-188 (corrected boundary) | near-universal, all 5 backends | -- (not separately checked) | 0.226 (**ordered**) | 0.613 (positive, but low-disorder undercuts a MoRF read) | **not a MoRF -- genuinely structured**, PROSITE boundary corrected above |
| DRB2 425-434 | 5/5-backend consensus | 0.699 (high) | 0.876 (disordered) | 0.942 (strong positive) | **cleanest genuine MoRF in either chain** |

Bottom line: this complex shows both phenomena the original question asked
about, cleanly separated by residue -- DRB4 273-282 and DRB2 425-434 are
real, evidence-convergent fold-upon-binding elements; DRB2 156-188 is real
structure but not a binding-induced one (a boundary-annotation gap, now
fixed above); and the broad helix content in some backends beyond these
specific hotspots (openfold3's 47%% long-run rate especially) is the part
still best explained as generic over-folding/hallucination rather than
biology, since it isn't backed by the same convergent sequence-predictor
signal.


## Does excluding the plausible genuine MoRF sites meaningfully reduce estimated over-folding?

Now that specific residue windows are believed to be real fold-upon-binding
sites rather than modeling artifacts (DRB4 268-284, structural onset through
the ANCHOR2-confirmed boundary; DRB2 425-434, the cleanest signal in either
chain), the natural next question: how much of the total "over-folding"
finding survives once those windows are set aside? If most of it evaporates,
the over-folding story was mostly real biology this whole time. If most of
it remains, there's still a real, unexplained excess folding problem beyond
what MoRF evidence accounts for.

**Gross** = %% helix over the full corrected disordered domain (as computed
throughout this notebook). **Net** = the same, over the disordered domain
with the plausible genuine MoRF window(s) removed.


In [38]:
MORF_WINDOWS = {"A": (268, 284), "B": (425, 434)}  # DRB2=A, DRB4=B
DISORDERED_BY_CHAIN = {"A": DRB2_DISORDERED, "B": DRB4_DISORDERED}
CHAIN_NAME = {"A": "DRB2", "B": "DRB4"}

dssp_bl = dssp.merge(backend_lookup, on=["fname", "cluster"], how="left").dropna(subset=["backend"])

def pct_helix_by_backend(chain, resnum_range):
    lo, hi = resnum_range
    sub = dssp_bl[(dssp_bl["chain"] == chain) & (dssp_bl["resnum"].between(lo, hi))]
    n_models_c = sub.groupby("backend").apply(lambda g: g[["fname","cluster"]].drop_duplicates().shape[0], include_groups=False)
    helix_n = sub[sub["ss_type"] == "helix"].groupby("backend").size()
    total_n = sub.groupby("backend").size()
    return (helix_n / total_n * 100).reindex(n_models_c.index, fill_value=0.0)

rows = []
for chain, (lo, hi) in DISORDERED_BY_CHAIN.items():
    morf_lo, morf_hi = MORF_WINDOWS[chain]
    gross = pct_helix_by_backend(chain, (lo, hi))
    # net window = disordered region with the MoRF window carved out (two
    # sub-ranges when the MoRF window sits strictly inside, one when it's
    # flush with either end)
    net_ranges = []
    if lo < morf_lo:
        net_ranges.append((lo, morf_lo - 1))
    if morf_hi < hi:
        net_ranges.append((morf_hi + 1, hi))
    net_parts = [pct_helix_by_backend(chain, r) for r in net_ranges]
    net = sum(net_parts) / len(net_parts) if len(net_parts) > 1 else net_parts[0]

    for b in gross.index:
        rows.append({"chain": CHAIN_NAME[chain], "backend": b,
                      "gross_pct_helix": gross[b], "net_pct_helix": net.get(b, float("nan"))})

reduction_df = pd.DataFrame(rows)
reduction_df["absolute_reduction_pp"] = reduction_df["gross_pct_helix"] - reduction_df["net_pct_helix"]
reduction_df["relative_reduction_pct"] = 100 * reduction_df["absolute_reduction_pp"] / reduction_df["gross_pct_helix"]
reduction_df = reduction_df.sort_values(["chain", "gross_pct_helix"], ascending=[True, False])
reduction_df.round(1)


,chain,backend,gross_pct_helix,net_pct_helix,absolute_reduction_pp,relative_reduction_pct
3,DRB2,openfold3,61.7,61.1,0.7,1.1
1,DRB2,boltz,23.0,24.4,-1.5,-6.4
2,DRB2,chai1,14.8,17.0,-2.2,-14.5
4,DRB2,protenix,8.6,10.7,-2.0,-23.8
0,DRB2,alphafold3,1.6,1.4,0.2,13.4
8,DRB4,openfold3,62.5,52.1,10.4,16.7
6,DRB4,boltz,26.8,21.2,5.6,20.9
7,DRB4,chai1,23.6,19.2,4.3,18.4
9,DRB4,protenix,3.9,4.7,-0.8,-21.5
5,DRB4,alphafold3,0.9,2.2,-1.3,-151.1


In [39]:
fig = go.Figure()
for chain_name, sub in reduction_df.groupby("chain"):
    fig.add_trace(go.Bar(x=sub["backend"], y=sub["gross_pct_helix"], name=f"{chain_name} gross (full disordered domain)",
                          marker_color="#C44E52", opacity=0.55, offsetgroup=chain_name, legendgroup=chain_name))
    fig.add_trace(go.Bar(x=sub["backend"], y=sub["net_pct_helix"], name=f"{chain_name} net (MoRF window excluded)",
                          marker_color="#4C72B0", opacity=0.85, offsetgroup=chain_name, legendgroup=chain_name,
                          base=0))
fig.update_layout(
    barmode="overlay", title="Gross vs. net (%% helix) over-folding estimate, by backend and chain",
    xaxis_title="backend", yaxis_title="%% helix in disordered domain",
    template=TEMPLATE, width=950, height=480,
)
save_fig(fig, "drb2_drb4_gross_vs_net_overfolding_by_backend.html", "fold_upon_binding")

print("Gross vs net %% helix, and the reduction once the plausible genuine MoRF window is excluded:\n")
print(reduction_df.round(1).to_string(index=False))
print(f"\nMean relative reduction across backend x chain rows: "
      f"{reduction_df['relative_reduction_pct'].mean():.1f}%%")


Saved: ../results/drb2_drb4/figures/domain_analysis/fold_upon_binding/drb2_drb4_gross_vs_net_overfolding_by_backend.html


Gross vs net %% helix, and the reduction once the plausible genuine MoRF window is excluded:

chain    backend  gross_pct_helix  net_pct_helix  absolute_reduction_pp  relative_reduction_pct
 DRB2  openfold3             61.7           61.1                    0.7                     1.1
 DRB2      boltz             23.0           24.4                   -1.5                    -6.4
 DRB2      chai1             14.8           17.0                   -2.2                   -14.5
 DRB2   protenix              8.6           10.7                   -2.0                   -23.8
 DRB2 alphafold3              1.6            1.4                    0.2                    13.4
 DRB4  openfold3             62.5           52.1                   10.4                    16.7
 DRB4      boltz             26.8           21.2                    5.6                    20.9
 DRB4      chai1             23.6           19.2                    4.3                    18.4
 DRB4   protenix              3.9         

### Verdict: no, not a significant decrease

The plausible genuine MoRF windows are small relative to the disordered
domains they sit in (DRB4 268-284 is 17/141 residues = 12%% of that domain;
DRB2 425-434 is 10/246 residues = 4%%), and it shows: excluding them barely
moves the aggregate over-folding estimate.

- **DRB2: no reduction at all.** For boltz, chai1 and protenix, net %% helix
  is *higher* than gross once 425-434 is excluded -- those backends'
  helix content is, if anything, *less* concentrated in the confirmed MoRF
  site than in the rest of the disordered region. openfold3 (the worst
  offender, 62%% gross) drops by only 0.7 points.
- **DRB4: a real but modest reduction**, ~17-21%% relative, for the three
  affected backends (openfold3 62.5%%->52.1%%, boltz 26.8%%->21.2%%, chai1
  23.6%%->19.2%%). Real, but four-fifths of each backend's over-folding
  remains after removing the one window we have good evidence is genuine
  biology.

So the MoRF-site findings from this investigation are real and worth
keeping, but they explain only a small slice of the total over-folding
phenomenon -- particularly for openfold3, whose ~50-62%% helix content in
both disordered domains is barely touched by excluding the confirmed real
site. The bulk of what's been called "over-folding" throughout this
notebook remains best attributed to backend-specific bias/hallucination,
not biology this investigation has been able to account for.


## Export per-cluster domain contact tables

In [43]:
for c in clusters:
    out_csv = out_path("per_cluster", f"drb2_drb4_domain_pair_counts_cluster{c}.csv")
    per_cluster_ct[c].to_csv(out_csv)
    print(f"Saved: {out_csv}")


Saved: ../results/drb2_drb4/figures/domain_analysis/per_cluster/drb2_drb4_domain_pair_counts_cluster1.csv
Saved: ../results/drb2_drb4/figures/domain_analysis/per_cluster/drb2_drb4_domain_pair_counts_cluster2.csv
